**Run evidence** · task `02e_purchase_order` · run `250170173800780` · result **SUCCESS** · executed 2026-10-19 17:09 UTC on serverless.

Exported from the Databricks job run with `_helper/export_run_evidence.py`. Source notebook: `02_build_gold/02e_purchase_order`.

# 02e — Workflow E → PURCHASE_ORDER (grain: EBELN + EBELP)

Consolidates the PO header/item/history into one analysis-ready PO line table.

**Steps (spec §4, Workflow E):**
1. Join `RAW_EKPO` to `RAW_EKKO` on EBELN, pulling LIFNR and EKORG onto the line.
2. Aggregate `RAW_EKBE` per EBELN+EBELP: take the **earliest goods-receipt**
   EVENT_DATE as GR_DATE (filter VGABE to the goods-receipt event type '1').
3. Derive `ON_TIME_FLAG` by comparing GR_DATE against the requested delivery date.
4. Output: one row per EBELN+EBELP with MATNR, LIFNR, EKORG, NETPR, GR_DATE,
   ON_TIME_FLAG (plus ORDER_DATE + REQUESTED_DATE, useful for OTD & Q8).

In [ ]:
%run ../00_config

[00_config] Using namespace: max_kohler_demo_catalog.purchasing_demo


DemoConfig(
  catalog       = 'max_kohler_demo_catalog'
  schema        = 'purchasing_demo'
  namespace     = 'max_kohler_demo_catalog.purchasing_demo'
  warehouse_id  = 'd7a3d1cfbba73c0a'
  model         = 'databricks-claude-sonnet-4-5'
  genie_space_1 = 'SAP Purchasing — Sourcing & Supplier Risk'
  genie_space_2 = 'SAP Purchasing — Purchase Order Analytics'
  mas_endpoint  = 'sap-purchasing-mas'
  app_name      = 'sap-purchasing-app'
  job_name      = 'sap_purchasing_lakeflow_pipeline'
  lakebase      = 'sap-purchasing-ops'/'purchasing_ops' (UC catalog 'sap_purchasing_ops')
)


In [ ]:
from pyspark.sql import functions as F

ekko = spark.table(CFG.fq("RAW_EKKO"))   # EBELN, LIFNR, EKORG, AEDAT
ekpo = spark.table(CFG.fq("RAW_EKPO"))   # EBELN, EBELP, MATNR, NETPR, EINDT
ekbe = spark.table(CFG.fq("RAW_EKBE"))   # EBELN, EBELP, VGABE, BEWTP, EVENT_DATE

VGABE_GOODS_RECEIPT = "1"   # goods-receipt event type

# Step 2: earliest GR date per PO line (filter to goods-receipt events only).
gr = (
    ekbe.filter(F.col("VGABE") == F.lit(VGABE_GOODS_RECEIPT))
        .groupBy("EBELN", "EBELP")
        .agg(F.min("EVENT_DATE").alias("GR_DATE"))
)

# Step 1: item joined to header for LIFNR/EKORG/order date.
po = (
    ekpo.join(ekko, on="EBELN", how="left")
        .join(gr, on=["EBELN", "EBELP"], how="left")
        .withColumnRenamed("AEDAT", "ORDER_DATE")
        .withColumnRenamed("EINDT", "REQUESTED_DATE")
        # Step 3: on-time when GR_DATE <= requested date. NULL GR_DATE = not yet received.
        .withColumn(
            "ON_TIME_FLAG",
            F.when(F.col("GR_DATE").isNull(), F.lit(None).cast("boolean"))
             .when(F.col("GR_DATE") <= F.col("REQUESTED_DATE"), F.lit(True))
             .otherwise(F.lit(False)),
        )
        .select(
            "EBELN", "EBELP", "MATNR", "LIFNR", "EKORG", "NETPR",
            "ORDER_DATE", "REQUESTED_DATE", "GR_DATE", "ON_TIME_FLAG",
        )
)

(po.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(CFG.fq("PURCHASE_ORDER")))

cnt = spark.table(CFG.fq("PURCHASE_ORDER")).count()
dupes = (spark.table(CFG.fq("PURCHASE_ORDER"))
         .groupBy("EBELN", "EBELP").count().filter("count > 1").count())
otd = spark.table(CFG.fq("PURCHASE_ORDER")).filter("ON_TIME_FLAG = true").count()
late = spark.table(CFG.fq("PURCHASE_ORDER")).filter("ON_TIME_FLAG = false").count()
print(f"PURCHASE_ORDER rows: {cnt:,} | grain violations: {dupes} | on-time: {otd:,} | late: {late:,}")
assert dupes == 0, "PURCHASE_ORDER grain violated — EBELN+EBELP not unique!"
display(spark.table(CFG.fq("PURCHASE_ORDER")).limit(10))

PURCHASE_ORDER rows: 711 | grain violations: 0 | on-time: 512 | late: 199


EBELN,EBELP,MATNR,LIFNR,EKORG,NETPR,ORDER_DATE,REQUESTED_DATE,GR_DATE,ON_TIME_FLAG
4500000001,10,MAT100020,V200011,2000,285.45,2026-07-12,2026-08-12,2026-08-11,True
4500000002,10,MAT100052,V200022,2000,521.63,2026-07-04,2026-08-09,2026-08-27,False
4500000002,20,MAT100020,V200022,2000,327.51,2026-07-04,2026-07-25,2026-08-19,False
4500000002,30,MAT100030,V200022,2000,423.39,2026-07-04,2026-08-09,2026-08-19,False
4500000003,10,MAT100005,V200021,1000,455.86,2026-09-08,2026-10-20,2026-10-15,True
4500000003,20,MAT100059,V200021,1000,435.03,2026-09-08,2026-10-05,2026-10-01,True
4500000004,10,MAT100039,V200001,2000,217.8,2026-08-09,2026-10-01,2026-09-30,True
4500000005,10,MAT100005,V200015,2000,426.31,2025-11-26,2025-12-13,2025-12-11,True
4500000006,10,MAT100034,V200009,2000,268.23,2025-11-04,2025-11-18,2025-11-24,False
4500000007,10,MAT100045,V200019,2000,383.31,2026-02-16,2026-03-20,2026-03-15,True


In [ ]:
dbutils.notebook.exit(f"02e_purchase_order: SUCCESS — {cnt} rows.")

Notebook exited: 02e_purchase_order: SUCCESS — 711 rows.
